# 6.10 XGBoost

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 6: Classification Algorithms

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-06-classification-algorithms/06.10_xgboost.ipynb)

Companion code for Section 6.10 of the book. Run the cells in order from top to bottom.

**Example 6.10.1**

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      stratify=y, random_state=42)

# ── XGBoost with evaluation set and early stopping ────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,           # Row subsampling (like Random Forest)
    colsample_bytree=0.8,    # Column subsampling per tree
    gamma=0.1,               # Minimum loss reduction to split
    reg_alpha=0.1,           # L1 regularisation
    reg_lambda=1.0,          # L2 regularisation
    use_label_encoder=False,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(X_train, y_train,
             eval_set=[(X_train, y_train), (X_test, y_test)],
             verbose=False)

print(f'Best iteration: {xgb_model.best_iteration}')
print(f'Train accuracy: {xgb_model.score(X_train, y_train):.4f}')
print(f'Test accuracy:  {xgb_model.score(X_test, y_test):.4f}')
print(f'Test AUC:       {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]):.4f}')

# ── Hyperparameter tuning ─────────────────────────────────────────
param_grid = {
    'max_depth':        [3, 4, 5],
    'learning_rate':    [0.01, 0.05, 0.1],
    'subsample':        [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

# Use RandomizedSearchCV for large param spaces
from sklearn.model_selection import RandomizedSearchCV
xgb_base = xgb.XGBClassifier(n_estimators=100, eval_metric='logloss',
                               use_label_encoder=False, random_state=42, verbosity=0)
rs = RandomizedSearchCV(xgb_base, param_grid, n_iter=20, cv=5,
                        scoring='roc_auc', n_jobs=-1, random_state=42)
rs.fit(X_train, y_train)
print(f'\nBest params: {rs.best_params_}')
print(f'Best CV AUC: {rs.best_score_:.4f}')
print(f'Test AUC:    {roc_auc_score(y_test, rs.predict_proba(X_test)[:,1]):.4f}')

# ── Feature importance (4 types) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('XGBoost Feature Importance', fontsize=14, fontweight='bold')

for ax, imp_type in zip(axes, ['weight', 'gain']):
    imp = xgb_model.get_booster().get_score(importance_type=imp_type)
    imp_df = pd.DataFrame(list(imp.items()), columns=['feature','importance'])
    imp_df['feature'] = imp_df['feature'].apply(
        lambda f: data.feature_names[int(f[1:])] if f.startswith('f') else f)
    imp_df = imp_df.sort_values('importance', ascending=True).tail(10)
    ax.barh(imp_df.feature, imp_df.importance, color='#FF6F00', edgecolor='white')
    ax.set_title(f'Importance Type: {imp_type}'); ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('xgboost_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# ── XGBoost key parameters guide ─────────────────────────────────
print('\n=== XGBoost Key Parameters ===')
params_guide = {
    'n_estimators':      'Number of trees (use early stopping to find optimal)',
    'learning_rate':     'Step size shrinkage: lower = more trees needed but better generalisation',
    'max_depth':         'Tree depth: 3-6 typical; deeper = more complexity + overfit risk',
    'subsample':         'Row sampling per tree: 0.7-0.9; reduces overfitting',
    'colsample_bytree':  'Feature sampling per tree: 0.7-0.9; reduces overfitting',
    'gamma':             'Min loss reduction to split: higher = more conservative',
    'reg_alpha':         'L1 regularisation on weights: induces sparsity',
    'reg_lambda':        'L2 regularisation on weights: prevents large weights',
    'scale_pos_weight':  'Class weight ratio for imbalanced data: neg/pos count',
}
for p, desc in params_guide.items():
    print(f'  {p:<22}: {desc}')